# Week 12 Lab — Water levels in the Pinal AMA: USGS data and the Pinal model

**HWRS 564a · Fall 2026**

In this lab you download every water-level measurement the USGS holds for the
Pinal Active Management Area (AMA), clean it, and measure how fast water levels
fell and then rose. Then you compare those rates with the Pinal County MODFLOW
model you loaded in Week 9.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without an error, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.

## Learning objectives

By the end of this notebook you can:

1. Download groundwater-level measurements for an area with
   `dataretrieval.waterdata.get_field_measurements`, with a saved copy to fall
   back on.
2. Describe a dataset before analyzing it: how many wells, how many
   measurements, and which years.
3. Use qualifier codes to remove measurements taken while a well was pumping.
4. Fit a linear trend to a water-level record, and write a function that does it
   for every well.
5. Compare measured trends with the Pinal model's calibration targets and
   simulated heads, and explain where they differ.

---

## Part 1 — Download the measurements

`dataretrieval` is the USGS's Python package for its water data. It builds the
web request, sends it, and returns a pandas DataFrame. We use its `waterdata`
module. (Many online tutorials use the older `nwis` module, which the USGS is
retiring.)

`BBOX` is the area to search: a box around Maricopa, Casa Grande, Eloy, and
Florence, given as west, south, east, north in degrees.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="Geopandas not installed")
warnings.filterwarnings("ignore", message="The program mf2005")

ROOT = Path("../..")
CACHE = ROOT / "data" / "cache"

BBOX = [-112.10, 32.60, -111.30, 33.15]      # west, south, east, north

### Every depth-to-water measurement in the box

`get_field_measurements` returns water levels measured by hand during site
visits. Parameter code `72019` is **depth to water below land surface**, in
feet. The cells below download them and keep six columns:

| Column | Meaning |
|---|---|
| `site` | the well's ID, such as `USGS-324637111335001` |
| `date` | the day of the measurement |
| `depth_to_water_ft` | depth from land surface down to the water, in feet |
| `qualifier` | notes on the measurement, such as `Static` or `Pumping` |
| `lon`, `lat` | the well's location |

The download takes 10–20 seconds.

In [ ]:
from dataretrieval import waterdata

raw, md = waterdata.get_field_measurements(bbox=BBOX, parameter_code="72019")
print("request:", md.url)
print(f"downloaded from the USGS: {len(raw):,} rows")

`raw` has many more columns than we need. Print their names, then look at the
first rows of the five we will use:

In [ ]:
print(list(raw.columns))
raw[["monitoring_location_id", "time", "value", "qualifier", "geometry"]].head()

Two of these columns need work before they fit in a table:

- `qualifier` holds lists, like `['Above', 'Pumping']`. The function `join_codes`
  turns each list into text, like `"Above, Pumping"`.
- `geometry` holds each well's location as a pair of numbers, longitude first.
  `.str[0]` takes the first number and `.str[1]` the second.

In [ ]:
def join_codes(q):
    """Qualifiers arrive as lists, like ['Above', 'Pumping']. Store them as text."""
    if isinstance(q, (list, tuple, np.ndarray)):
        return ", ".join(str(x) for x in q)
    return "" if q is None else str(q)


join_codes(["Above", "Pumping"])

Now build a new table with the six columns. `time` arrives as text with a time
zone; the `date` line converts it to dates and drops the time of day.

In [ ]:
levels = pd.DataFrame({
    "site": raw["monitoring_location_id"],
    "date": pd.to_datetime(raw["time"], utc=True, format="mixed")
              .dt.tz_localize(None).dt.normalize(),
    "depth_to_water_ft": pd.to_numeric(raw["value"], errors="coerce"),
    "qualifier": raw["qualifier"].apply(join_codes),
    "lon": raw["geometry"].str[0].round(5),
    "lat": raw["geometry"].str[1].round(5),
})
levels = levels.sort_values(["site", "date"], ignore_index=True)
levels.head()

### If the download fails

Websites go down, and a whole class downloading at once can slow one down. If
any of the cells above gave an error, use the saved copy of the same table in
`data/cache/` instead: remove the `#` at the start of each line below (select
the lines and press Ctrl+/ or Cmd+/), then run the cell. Otherwise skip it.

In [ ]:
# levels = pd.read_csv(CACHE / "waterdata_pinal_field_72019.csv",
#                      parse_dates=["date"], keep_default_na=False,
#                      na_values={"depth_to_water_ft": [""]})
# print(f"saved copy: {len(levels):,} rows")

Last, add a `year` column and look at the finished table.

In [ ]:
levels["year"] = levels["date"].dt.year + (levels["date"].dt.dayofyear - 1) / 365.25
levels.head()

Look at the IDs. Most start with `USGS-`, but some start with `AZ014-`. That
prefix is the agency code for the Arizona Department of Water Resources (ADWR):
the USGS database also stores some of ADWR's measurements. The prefix is part of
the ID, so `USGS-...` and `AZ014-...` with the same number are different
records.

The `year` column is the date as a decimal year (1985.5 is the middle of 1985).
It makes fitting trends easier later.

### YOUR TURN 1

Before analyzing anything, describe what you downloaded:

- `n_wells` — the number of different wells
- `n_measurements` — the number of rows
- `first_year`, `last_year` — the calendar years of the earliest and latest
  measurements, as integers

In [ ]:
# YOUR TURN
n_wells = ...
n_measurements = ...
first_year = ...
last_year = ...

In [ ]:
# CHECK
assert n_wells == levels["site"].nunique(), f"got {n_wells}"
assert n_measurements == len(levels), f"got {n_measurements}"
assert first_year == levels["date"].dt.year.min(), f"got {first_year}"
assert last_year == levels["date"].dt.year.max(), f"got {last_year}"
assert n_wells > 2000, f"expected more than 2,000 wells, got {n_wells}"
print(f"{n_measurements:,} measurements at {n_wells:,} wells, {first_year} to {last_year}. Correct.")

Now look at **when** the measurements were made and **where** the wells are.

In [ ]:
per_decade = levels.groupby((levels["date"].dt.year // 10) * 10).size()

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.2))
left.bar(per_decade.index, per_decade.values, width=8, color="#0C234B")
left.set_xlabel("decade")
left.set_ylabel("measurements")
left.set_title("Measurements per decade")

wells = levels.groupby("site")[["lon", "lat"]].first()
n_per_well = levels.groupby("site").size()
right.scatter(wells["lon"], wells["lat"], s=np.clip(n_per_well, 1, 60), alpha=0.4,
              color="#AB0520")
right.set_xlabel("longitude")
right.set_ylabel("latitude")
right.set_title("Wells (dot size = number of measurements)")
right.set_aspect(1 / np.cos(np.radians(32.9)))
plt.tight_layout()
plt.show()

print(per_decade.to_string())

Most measurements are from the 1940s through the 1990s, and the USGS database
has none in this area after 2013. That doesn't mean nobody is measuring: ADWR
now does most of the monitoring in Pinal County and keeps the results in its
[Groundwater Site Inventory (GWSI)](https://azwatermaps.azwater.gov/gwsiweb/).
Always check whether the agency you're downloading from is still the one
collecting the data.

---

## Part 2 — Which measurements to keep

The `qualifier` column records the conditions when each measurement was made.

In [ ]:
levels["qualifier"].value_counts()

| Qualifier | Meaning |
|---|---|
| `Static` | a normal measurement: the well wasn't pumping |
| `Above, Pumping` | the well was pumping, or had been pumping recently |
| `NoMeasurement, Dry` | the well was dry, so there is no depth |
| `ForeignSubstance` | something in the well (such as oil) affected the reading |
| (blank) | no qualifier was recorded |

A pumping well pulls the water level down around itself, so a measurement taken
during pumping is deeper than the water table nearby. At wells that have both
kinds of measurement, compare them:

In [ ]:
both = (levels[levels["qualifier"].isin(["Static", "Above, Pumping"])]
        .groupby(["site", "qualifier"])["depth_to_water_ft"].median()
        .unstack().dropna())
difference = both["Above, Pumping"] - both["Static"]
print(f"{len(both)} wells have both kinds of measurement")
print(f"median difference (pumping minus static): {difference.median():.0f} ft deeper")

Pumping measurements are typically about 55 ft deeper. Mixing them in with
static measurements adds large errors that have nothing to do with the
aquifer's water level.

### YOUR TURN 2

Build `static`: only the rows whose `qualifier` is exactly `"Static"` **and**
that have a depth value.

In [ ]:
# YOUR TURN
static = ...

In [ ]:
# CHECK
assert (static["qualifier"] == "Static").all(), "some rows are not Static"
assert static["depth_to_water_ft"].notna().all(), "some rows have no depth"
expected = int(((levels["qualifier"] == "Static") & levels["depth_to_water_ft"].notna()).sum())
assert len(static) == expected, f"expected {expected} rows, got {len(static)}"
print(f"kept {len(static):,} of {len(levels):,} measurements "
      f"({len(static) / len(levels):.0%}). Correct.")

---

## Part 3 — One well

Start with the well that has the most static measurements, near Eloy. Depth to
water is measured downward, so the plot flips the y-axis: a rising water table
goes up on the page.

In [ ]:
WELL = "USGS-324637111335001"
one = static[static["site"] == WELL].sort_values("date")
print(f"{WELL}: {len(one)} static measurements, {one['date'].min():%Y} to {one['date'].max():%Y}")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(one["date"], one["depth_to_water_ft"], "o-", ms=3, lw=0.8, color="#0C234B")
ax.invert_yaxis()
ax.set_ylabel("depth to water (ft)")
ax.set_title(f"{WELL}, near Eloy")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The water level fell through the 1970s and rose quickly after the mid-1980s. To
put numbers on that, fit a straight line to each period. `np.polyfit(x, y, 1)`
fits a line and returns `[slope, intercept]`, so `np.polyfit(x, y, 1)[0]` is
the slope.

One sign detail: when the water table **rises**, depth to water **decreases**.
We want rising to be positive, so the rate is **minus** the slope of depth
against time.

### YOUR TURN 3

Using `one`, compute two rates in feet per year, positive when the water table
rises:

- `decline_rate` — for measurements with `year` from 1965 up to (not
  including) 1980
- `recovery_rate` — for `year` from 1985 up to (not including) 2005

In [ ]:
# YOUR TURN
decline_rate = ...
recovery_rate = ...

In [ ]:
# CHECK
assert decline_rate < 0, f"the water table was falling, so the rate should be negative; got {decline_rate:.2f}"
assert abs(decline_rate - (-2.54)) < 0.1, f"expected about -2.54 ft/yr, got {decline_rate:.2f}"
assert abs(recovery_rate - 11.34) < 0.2, f"expected about +11.34 ft/yr, got {recovery_rate:.2f}"
print(f"1965-1980: {decline_rate:+.2f} ft/yr")
print(f"1985-2005: {recovery_rate:+.2f} ft/yr. Correct.")

---

## Part 4 — Every well

One well isn't the basin. Now compute a trend for every well with enough data.
A trend from two measurements, or from measurements a year apart, isn't
reliable, so a well needs **at least 5 measurements spanning at least 10 years**
within the period.

### YOUR TURN 4

Write `water_level_trend(df, value_col, start, end, min_n=5, min_span=10)`. It
should:

1. keep rows of `df` with `year` from `start` up to (not including) `end`;
2. for each `site`, skip it unless it has at least `min_n` rows and its years
   span at least `min_span` years;
3. otherwise fit a line of `value_col` against `year` and record the slope;
4. return a Series of slopes indexed by site.

It returns the plain slope of whatever column you give it. You handle the sign
afterward, because depth and head have opposite signs.

In [ ]:
def water_level_trend(df, value_col, start, end, min_n=5, min_span=10):
    """Slope of value_col against year, per site, for sites with enough data."""
    # YOUR TURN
    ...

In [ ]:
# CHECK
toy = pd.DataFrame({
    "site": ["a"] * 5 + ["b"] * 4 + ["c"] * 5 + ["d"] * 6,
    "year": [1990, 1993, 1996, 1999, 2002,  1990, 1995, 2000, 2004,
             1990, 1992, 1994, 1996, 1998,  1985, 1988, 1991, 1994, 1997, 2000],
})
toy["value"] = np.where(toy["site"] == "d", 50 - 1.5 * (toy["year"] - 1985),
                        100 + 2.0 * (toy["year"] - 1990))
result = water_level_trend(toy, "value", 1980, 2010)
assert isinstance(result, pd.Series), f"should return a Series, got {type(result)}"
assert set(result.index) == {"a", "d"}, (
    f"expected sites a and d only, got {sorted(result.index)} "
    "(b has 4 measurements; c spans only 8 years)")
assert abs(result["a"] - 2.0) < 1e-9 and abs(result["d"] + 1.5) < 1e-9, result.to_dict()
assert set(water_level_trend(toy, "value", 1980, 2001).index) == {"d"}, \
    "rows outside start-end should be ignored (site a has only 4 rows before 2001)"
print(result.to_string())
print("Correct.")

Apply it to the static measurements for two periods: the **decline** from 1950
to 1975, when groundwater pumping for agriculture was at its peak, and the
**recovery** from 1985 to 2005. Remember the minus sign for depth.

In [ ]:
ERAS = {"decline 1950-75": (1950, 1975), "recovery 1985-2005": (1985, 2005)}

usgs = {era: -water_level_trend(static, "depth_to_water_ft", a, b)
        for era, (a, b) in ERAS.items()}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (era, rates) in zip(axes, usgs.items()):
    ax.hist(rates, bins=30, color="#0C234B", alpha=0.85)
    ax.axvline(0, color="black", lw=1)
    ax.axvline(rates.median(), color="#AB0520", lw=2, ls="--",
               label=f"median {rates.median():+.1f} ft/yr")
    ax.set_title(f"{era}: {len(rates)} wells")
    ax.set_xlabel("water-level change (ft/yr, rising positive)")
    ax.legend(frameon=False)
axes[0].set_ylabel("wells")
plt.tight_layout()
plt.show()

Water levels fell at nearly every well from 1950 to 1975 and rose at most wells
from 1985 to 2005.

How much did removing the pumping measurements matter? Repeat the recovery
calculation with **all** measurements:

In [ ]:
with_depth = levels[levels["depth_to_water_ft"].notna()]
all_rates = -water_level_trend(with_depth, "depth_to_water_ft", 1985, 2005)
print(f"recovery 1985-2005, static only:      median {usgs['recovery 1985-2005'].median():+.2f} ft/yr")
print(f"recovery 1985-2005, all measurements: median {all_rates.median():+.2f} ft/yr")

Including the pumping measurements makes the recovery look about a third
faster. In 1985–89, 13% of measurements were taken while pumping; after 1995,
only 2–3% were. Deep readings early in the period and normal readings later make
the water level look like it rose more than it did.
Cleaning the data changed the answer.

---

## Part 5 — Compare with the Pinal County model

ADWR calibrated the Pinal County model against measured water levels. For each
measurement, the model records the head it simulated at the same place and
time. Those pairs are the model's **head observations** (the HOB package you saw
in Week 9).

The cell below reads them from your copy of the model if it can find it
(`pinal_county_model/` at the top of the repo, or in `labs/week09/`). Otherwise
it uses a saved copy. Each row is one observation:

| Column | Meaning |
|---|---|
| `site` | ADWR's ID for the observation well |
| `row`, `col` | the model cell (starting at 0) |
| `year` | the time of the observation, as a decimal year |
| `observed` | the measured head, in ft above sea level |
| `simulated` | the head the model simulated there and then |

These are **heads** (water-level elevations), not depths, so a rising water
table already has a positive slope. No minus sign this time.

In [ ]:
MODEL_DIRS = [ROOT / "pinal_county_model", ROOT / "labs" / "week09" / "pinal_county_model"]
NAM_NAME = "PM_AWS2019_SS25115.nam"


def read_model_obs(ws):
    """Observed and simulated heads from the model's HOB input and output files."""
    import flopy
    nam = ws / NAM_NAME
    units = {}
    for line in nam.read_text(errors="replace").splitlines():
        w = line.split()
        if len(w) >= 3 and not line.lstrip().startswith("#"):
            units[int(w[1])] = (w[0].upper(), w[2])
    hob_file = next(f for t, f in units.values() if t == "HOB")
    lines = [l for l in (ws / hob_file).read_text(errors="replace").splitlines()
             if l.strip() and not l.lstrip().startswith("#")]
    out_file = units[int(lines[0].split()[3])][1]
    recs = []
    for l in lines[2:]:
        w = l.split()
        if len(w) >= 9 and not w[0][0].isdigit():
            recs.append({"name": w[0], "row": int(w[2]) - 1, "col": int(w[3]) - 1,
                         "sp": int(w[4]), "toffset": float(w[5])})
    obs = pd.DataFrame(recs).merge(
        pd.read_csv(ws / out_file, sep=r"\s+", skiprows=1,
                    names=["simulated", "observed", "name"]), on="name")
    m = flopy.modflow.Modflow.load(NAM_NAME, model_ws=str(ws), load_only=["dis"],
                                   check=False, forgive=True, verbose=False)
    perlen = m.dis.perlen.array
    # stress period 1 is steady-state 1922; stress period n >= 2 is year 1921 + n
    obs["year"] = np.where(obs["sp"] == 1, 1922.0,
                           1921 + obs["sp"] + obs["toffset"] / perlen[obs["sp"] - 1])
    obs["site"] = obs["name"].str[:-5]
    return obs[["site", "row", "col", "year", "observed", "simulated"]]


model_ws = next((d for d in MODEL_DIRS if (d / NAM_NAME).exists()), None)
if model_ws is not None:
    model_obs = read_model_obs(model_ws)
    print(f"read from {model_ws.relative_to(ROOT)}: {len(model_obs):,} observations")
else:
    model_obs = pd.read_csv(CACHE / "pinal_model_head_obs.csv")
    print(f"model folder not found; saved copy: {len(model_obs):,} observations")
model_obs.head()

One limitation before comparing: the model files don't include map coordinates
for the grid, and ADWR's site IDs don't match USGS IDs. So we can't match USGS
wells to model cells one by one. Instead, we compare the **typical** (median)
trend across each set of wells, for the same area and the same periods.

### YOUR TURN 5

Build `comparison`: a DataFrame of median trends in ft/yr, with

- **index**: `"USGS wells (static)"`, `"model targets (observed)"`,
  `"model (simulated)"`
- **columns**: the two keys of `ERAS`, `"decline 1950-75"` and
  `"recovery 1985-2005"`

The USGS row comes from `usgs`. For the model rows, apply `water_level_trend`
to `model_obs`, using the `observed` and `simulated` columns.

In [ ]:
# YOUR TURN
comparison = ...

In [ ]:
# CHECK
rows = ["USGS wells (static)", "model targets (observed)", "model (simulated)"]
assert list(comparison.index) == rows, f"index should be {rows}, got {list(comparison.index)}"
assert list(comparison.columns) == list(ERAS), f"columns should be {list(ERAS)}, got {list(comparison.columns)}"
expected = pd.DataFrame([[-4.73, 3.60], [-4.86, 3.63], [-4.45, 2.30]], index=rows, columns=list(ERAS))
diff = (comparison - expected).abs()
assert (diff < 0.3).all().all(), f"some values are off:\n{comparison.round(2)}"
print(comparison.round(2).to_string())
print("Correct.")

In [ ]:
ax = comparison.T.plot.bar(figsize=(8, 3.8), rot=0,
                           color=["#0C234B", "#81D3EB", "#AB0520"])
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("median water-level change (ft/yr)")
ax.legend(frameon=False)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

**What the comparison shows:**

- During the **decline**, all three agree: about 4.5–5 ft per year of falling
  water levels. The model reproduces this period well.
- During the **recovery**, the USGS static measurements and ADWR's calibration
  targets agree (about +3.6 ft/yr), even though they come from different
  databases. That's a useful check that the targets are consistent with other
  data.
- The model's **simulated** recovery is slower, about +2.3 ft/yr, or roughly
  two-thirds of what was measured.

**Question 1.** List two things in a groundwater model that could make it
simulate a slower recovery than was measured. Think about the inputs (pumping,
recharge) and the aquifer properties.

*Your answer:*

**Question 2.** Project 3 uses this model to predict water levels to 2115. If
the model under-predicts how fast water levels rose when pumping decreased,
what does that suggest about its predictions for future scenarios that reduce
pumping?

*Your answer:*

**Question 3.** The model cells are 2,640 ft (half a mile) on a side. How might
that affect a comparison with water levels measured in individual wells?

*Your answer:*

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

**HW 6 — Build, audit, and evaluate a groundwater dataset**, Wednesday 11/18 at
11:59pm, through D2L. Parts 1–4 of this notebook practice the same steps:
download, describe, clean, and summarize.

## Stuck?

- **The download hangs or fails:** interrupt the kernel if it hangs, then run
  the cell under *If the download fails* in Part 1. The saved copy should give
  the same answers.
- **`static` keeps the wrong rows:** check that you compared `qualifier` to
  `"Static"` and also removed rows where `depth_to_water_ft` is missing
  (`.notna()`).
- **Your rates have the wrong sign:** depth to water gets smaller when the water
  table rises. Put a minus sign in front of the depth slope; heads don't need one.
- **`water_level_trend` returns an empty Series:** check the year filter. Rows
  should satisfy `year >= start` and `year < end`.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 232.